# Sentiment Analysis AI
This notebook demonstrates sentiment analysis using multiple approaches: VADER (lexicon-based), TextBlob, and Transformers (BERT)

## 1. Install Required Libraries

In [ ]:
# Install required packages
!pip install nltk textblob transformers torch pandas matplotlib seaborn scikit-learn

## 2. Import Libraries

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from transformers import pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

# Download required NLTK data
nltk.download('vader_lexicon')
nltk.download('punkt')

print("Libraries imported successfully!")

## 3. Initialize Sentiment Analysis Models

In [ ]:
# Initialize VADER sentiment analyzer
vader_analyzer = SentimentIntensityAnalyzer()

# Initialize Hugging Face transformer pipeline (distilbert for faster inference)
transformer_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("Models initialized successfully!")

## 4. Define Sentiment Analysis Functions

In [ ]:
def analyze_sentiment_vader(text):
    """
    Analyze sentiment using VADER (Valence Aware Dictionary and sEntiment Reasoner)
    Best for social media and short texts
    """
    scores = vader_analyzer.polarity_scores(text)
    compound = scores['compound']
    
    if compound >= 0.05:
        sentiment = 'POSITIVE'
    elif compound <= -0.05:
        sentiment = 'NEGATIVE'
    else:
        sentiment = 'NEUTRAL'
    
    return {
        'text': text,
        'sentiment': sentiment,
        'compound_score': compound,
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral': scores['neu']
    }

def analyze_sentiment_textblob(text):
    """
    Analyze sentiment using TextBlob
    Simple and intuitive approach
    """
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    if polarity > 0.1:
        sentiment = 'POSITIVE'
    elif polarity < -0.1:
        sentiment = 'NEGATIVE'
    else:
        sentiment = 'NEUTRAL'
    
    return {
        'text': text,
        'sentiment': sentiment,
        'polarity': polarity,
        'subjectivity': subjectivity
    }

def analyze_sentiment_transformer(text):
    """
    Analyze sentiment using Transformer (BERT-based)
    More accurate but slower
    """
    result = transformer_analyzer(text[:512])[0]  # Truncate to 512 tokens
    
    return {
        'text': text[:100] + '...' if len(text) > 100 else text,
        'sentiment': result['label'],
        'confidence': round(result['score'], 4)
    }

print("Sentiment analysis functions defined!")

## 5. Sample Dataset

In [ ]:
# Sample texts for sentiment analysis
sample_texts = [
    "I absolutely love this product! It's amazing and exceeded my expectations.",
    "This is terrible. I'm very disappointed with the quality.",
    "The weather is nice today.",
    "I had a wonderful time at the restaurant. The food was delicious!",
    "Worst experience ever. Terrible service and overpriced.",
    "It's an okay product, nothing special but gets the job done.",
    "I'm so happy! Best day ever!",
    "This is frustrating and annoying. Very disappointed.",
    "The movie was entertaining and well-produced.",
    "I don't like this at all. Complete waste of money."
]

print(f"Sample dataset created with {len(sample_texts)} texts")

## 6. Analyze with VADER

In [ ]:
# Analyze using VADER
vader_results = [analyze_sentiment_vader(text) for text in sample_texts]
vader_df = pd.DataFrame(vader_results)

print("\n=== VADER Sentiment Analysis ===")
print(vader_df[['text', 'sentiment', 'compound_score']].to_string())

## 7. Analyze with TextBlob

In [ ]:
# Analyze using TextBlob
textblob_results = [analyze_sentiment_textblob(text) for text in sample_texts]
textblob_df = pd.DataFrame(textblob_results)

print("\n=== TextBlob Sentiment Analysis ===")
print(textblob_df[['text', 'sentiment', 'polarity']].to_string())

## 8. Analyze with Transformer (BERT)

In [ ]:
# Analyze using Transformer
print("\n=== Transformer (BERT) Sentiment Analysis ===")
print("Processing... This may take a moment on first run.\n")

transformer_results = []
for i, text in enumerate(sample_texts, 1):
    result = analyze_sentiment_transformer(text)
    transformer_results.append(result)
    print(f"{i}. {result['sentiment']} (confidence: {result['confidence']})")

transformer_df = pd.DataFrame(transformer_results)

## 9. Comparison Visualization

In [ ]:
# Compare sentiment distribution across methods
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# VADER
vader_counts = vader_df['sentiment'].value_counts()
axes[0].bar(vader_counts.index, vader_counts.values, color=['green', 'red', 'gray'])
axes[0].set_title('VADER Sentiment Distribution')
axes[0].set_ylabel('Count')

# TextBlob
textblob_counts = textblob_df['sentiment'].value_counts()
axes[1].bar(textblob_counts.index, textblob_counts.values, color=['green', 'red', 'gray'])
axes[1].set_title('TextBlob Sentiment Distribution')
axes[1].set_ylabel('Count')

# Transformer
transformer_counts = transformer_df['sentiment'].value_counts()
axes[2].bar(transformer_counts.index, transformer_counts.values, color=['green', 'red'])
axes[2].set_title('Transformer Sentiment Distribution')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\nSentiment distribution visualized!")

## 10. VADER Score Distribution

In [ ]:
# Visualize VADER compound scores
plt.figure(figsize=(12, 6))

colors = ['green' if score > 0.05 else 'red' if score < -0.05 else 'gray' 
          for score in vader_df['compound_score']]

plt.bar(range(len(vader_df)), vader_df['compound_score'], color=colors)
plt.axhline(y=0.05, color='r', linestyle='--', label='Positive threshold')
plt.axhline(y=-0.05, color='b', linestyle='--', label='Negative threshold')
plt.xlabel('Sample Index')
plt.ylabel('Compound Score')
plt.title('VADER Compound Sentiment Scores')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 11. TextBlob Polarity vs Subjectivity

In [ ]:
# Scatter plot of polarity vs subjectivity
plt.figure(figsize=(10, 6))

colors = ['green' if sentiment == 'POSITIVE' else 'red' if sentiment == 'NEGATIVE' else 'gray'
          for sentiment in textblob_df['sentiment']]

plt.scatter(textblob_df['polarity'], textblob_df['subjectivity'], c=colors, s=200, alpha=0.6)
plt.xlabel('Polarity (Sentiment Strength)')
plt.ylabel('Subjectivity (Opinion vs Fact)')
plt.title('TextBlob: Polarity vs Subjectivity Analysis')
plt.grid(True, alpha=0.3)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.axhline(y=0.5, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='Positive'),
                   Patch(facecolor='red', label='Negative'),
                   Patch(facecolor='gray', label='Neutral')]
plt.legend(handles=legend_elements)

plt.show()

## 12. Detailed Analysis Report

In [ ]:
# Create a detailed comparison report
comparison_df = pd.DataFrame({
    'Text': [text[:50] + '...' if len(text) > 50 else text for text in sample_texts],
    'VADER': vader_df['sentiment'],
    'TextBlob': textblob_df['sentiment'],
    'Transformer': transformer_df['sentiment'],
    'VADER Score': vader_df['compound_score'].round(3),
    'TextBlob Polarity': textblob_df['polarity'].round(3),
    'Transformer Conf': transformer_df['confidence']
})

print("\n=== Detailed Comparison Report ===")
print(comparison_df.to_string(index=False))

## 13. Custom Text Analysis

In [ ]:
def analyze_custom_text(text):
    """
    Analyze custom user-provided text with all three methods
    """
    print(f"\nAnalyzing: '{text}'\n")
    
    # VADER
    vader_result = analyze_sentiment_vader(text)
    print(f"VADER Result:")
    print(f"  Sentiment: {vader_result['sentiment']}")
    print(f"  Compound Score: {vader_result['compound_score']:.4f}")
    print(f"  Positive: {vader_result['positive']:.4f}, Negative: {vader_result['negative']:.4f}, Neutral: {vader_result['neutral']:.4f}\n")
    
    # TextBlob
    textblob_result = analyze_sentiment_textblob(text)
    print(f"TextBlob Result:")
    print(f"  Sentiment: {textblob_result['sentiment']}")
    print(f"  Polarity: {textblob_result['polarity']:.4f}")
    print(f"  Subjectivity: {textblob_result['subjectivity']:.4f}\n")
    
    # Transformer
    transformer_result = analyze_sentiment_transformer(text)
    print(f"Transformer Result:")
    print(f"  Sentiment: {transformer_result['sentiment']}")
    print(f"  Confidence: {transformer_result['confidence']:.4f}")

# Example usage
analyze_custom_text("This product is fantastic and I love it!")

## 14. Batch Processing with Custom Input

In [ ]:
# Example: Analyze your own texts
custom_texts = [
    "I'm really happy with my purchase!",
    "This is not what I expected.",
    "The service was average."
]

print("\n=== Analyzing Custom Texts ===")
custom_results = []

for text in custom_texts:
    vader = analyze_sentiment_vader(text)
    custom_results.append({
        'Text': text,
        'Sentiment': vader['sentiment'],
        'Score': vader['compound_score']
    })

custom_df = pd.DataFrame(custom_results)
print(custom_df.to_string(index=False))

## 15. Summary and Recommendations

In [ ]:
print("\n" + "="*60)
print("SENTIMENT ANALYSIS METHODS COMPARISON")
print("="*60)

summary = {
    'Method': ['VADER', 'TextBlob', 'Transformer (BERT)'],
    'Speed': ['Very Fast', 'Fast', 'Slow (GPU Recommended)'],
    'Accuracy': ['Good', 'Fair', 'Excellent'],
    'Best For': ['Social Media, Short Texts', 'General Text', 'Production, High Accuracy'],
    'Resource Usage': ['Low', 'Low', 'High'],
    'Language Support': ['English', 'Multiple', 'Multiple']
}

summary_df = pd.DataFrame(summary)
print("\n" + summary_df.to_string(index=False))

print("\n" + "="*60)
print("RECOMMENDATIONS:")
print("="*60)
print("1. Use VADER for: Real-time analysis, social media, tweets")
print("2. Use TextBlob for: Quick sentiment checks, learning purposes")
print("3. Use Transformer for: Production systems, high accuracy needed")
print("="*60)